# Rol 3 - Dia 1: seleccion por torneo

Responsabilidad exclusiva del Rol 3 en un equipo de cinco integrantes.
El examen pide para el Dia 1 seleccion por torneo y compatibilidad con la tabla de genes.
Cruces y mutaciones quedan pendientes del Dia 2; no se implementan tareas del Rol 5.

Se reviso `ag_ejemplo_reinas.ipynb` del docente, seccion 4, celda de seleccion:
`random.sample(poblacion, k)` seguido de `max(competidores, key=fitness)`.
Conservamos sorteo sin reemplazo y maximizacion. Adaptamos las permutaciones a
vectores continuos, usamos `numpy.random.Generator`, recibimos aptitudes ya
calculadas y devolvemos una copia. La implementacion se importa desde `src`.

Cada torneo favorece a los mejores participantes, pero con N > k no garantiza
seleccionar el mejor de toda la poblacion. En empates gana el primero sorteado.
Un individuo puede ganar varios torneos. No se reparan ni modifican los genes.

In [1]:
from pathlib import Path
import sys
import numpy as np

RAIZ = Path.cwd().resolve()
if not (RAIZ / "src" / "representacion.py").is_file():
    RAIZ = RAIZ.parent
if not (RAIZ / "src" / "representacion.py").is_file():
    raise FileNotFoundError("Ejecuta desde la raiz del repositorio o notebooks/.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.operadores import seleccion_torneo
from src.representacion import (
    num_genes, generar_poblacion_inicial, validar_individuo,
)

D = num_genes()
print(f"Genes actuales: {D}")

Genes actuales: 8


## 1. Maximizacion y copia independiente

Las aptitudes de este ejemplo son artificiales para aislar el operador.
Tambien se admiten aptitudes negativas: solo importa su orden.
Con N = k = 3 participan todos y siempre debe ganar el maximo.

In [2]:
poblacion = np.array([np.full(D, valor) for valor in (0.2, 0.5, 0.8)])
aptitudes = np.array([-3.0, 2.0, -1.0])
poblacion_original = poblacion.copy()
aptitudes_originales = aptitudes.copy()
rng = np.random.default_rng(42)

for _ in range(20):
    ganador = seleccion_torneo(poblacion, aptitudes, k=3, rng=rng)
    assert np.array_equal(ganador, poblacion[1])
    assert not np.shares_memory(ganador, poblacion)

assert np.array_equal(poblacion, poblacion_original)
assert np.array_equal(aptitudes, aptitudes_originales)
ganador[0] = 0.0
assert np.array_equal(poblacion, poblacion_original)
assert np.array_equal(aptitudes, aptitudes_originales)
print("Maximo en 20 torneos, entradas intactas y copia independiente: OK")

Maximo en 20 torneos, entradas intactas y copia independiente: OK


## 2. Reproducibilidad, empates y compatibilidad

El generador se crea una sola vez por secuencia y se comparte entre llamadas.
Dos generadores con igual semilla y las mismas operaciones reproducen el sorteo.
La prueba de empates comprueba el orden sorteado, no el indice de la poblacion.

In [3]:
poblacion_aleatoria = generar_poblacion_inicial(10, rng=np.random.default_rng(2024))
aptitudes_ejemplo = np.arange(10, dtype=float)
rng_a = np.random.default_rng(7)
rng_b = np.random.default_rng(7)
secuencia_a = [seleccion_torneo(poblacion_aleatoria, aptitudes_ejemplo, rng=rng_a) for _ in range(20)]
secuencia_b = [seleccion_torneo(poblacion_aleatoria, aptitudes_ejemplo, rng=rng_b) for _ in range(20)]
assert np.array_equal(secuencia_a, secuencia_b)

# Contrastar el sorteo y su avance con un generador independiente.
rng_control = np.random.default_rng(7)
for ganador in secuencia_a:
    indices = rng_control.choice(10, size=3, replace=False)
    assert len(np.unique(indices)) == 3
    assert np.array_equal(ganador, poblacion_aleatoria[indices[np.argmax(aptitudes_ejemplo[indices])]])
    valido, mensaje = validar_individuo(ganador)
    assert valido, mensaje
    assert ganador.shape == (D,) and ganador.dtype == np.float64

rng_empate = np.random.default_rng(101)
rng_control = np.random.default_rng(101)
indices = rng_control.choice(10, size=3, replace=False)
ganador_empate = seleccion_torneo(poblacion_aleatoria, np.zeros(10), rng=rng_empate)
assert np.array_equal(ganador_empate, poblacion_aleatoria[indices[0]])

# k=1, entero NumPy y generador omitido tambien son validos.
ganador_unico = seleccion_torneo(poblacion[:1], [-5.0], k=np.int64(1))
assert np.array_equal(ganador_unico, poblacion[0])
print("Reproducibilidad, muestreo, empates, k=1 y compatibilidad float64: OK")

Reproducibilidad, muestreo, empates, k=1 y compatibilidad float64: OK


## 3. Entradas invalidas

Se informa ValueError sin reducir k ni reparar genes. La correspondencia
semantica entre fitness[i] y poblacion[i] es responsabilidad del llamador;
el operador puede comprobar la longitud, pero no detectar una permutacion
incorrecta de aptitudes ya calculadas.

In [4]:
def comprobar_value_error(pob, fits, k=3):
    try:
        seleccion_torneo(pob, fits, k=k, rng=np.random.default_rng(42))
    except ValueError:
        return
    raise AssertionError("Se esperaba ValueError")

casos_invalidos = [
    (poblacion[:2], aptitudes[:2], 3),
    (poblacion, aptitudes[:2], 3),
    (poblacion, aptitudes.reshape(3, 1), 3),
    (poblacion[0], aptitudes, 3),
    (np.empty((0, D)), np.array([]), 1),
    (np.zeros((3, D + 1)), aptitudes, 3),
]
for valor in (np.nan, np.inf, -np.inf):
    fits_invalidos = aptitudes.copy()
    fits_invalidos[0] = valor
    casos_invalidos.append((poblacion, fits_invalidos, 3))
for valor in (-0.01, 1.01, np.nan, np.inf):
    pob_invalida = poblacion.copy()
    pob_invalida[0, 0] = valor
    casos_invalidos.append((pob_invalida, aptitudes, 3))
for k_invalido in (0, -1, 4, 2.5, 3.0, True, np.bool_(False)):
    casos_invalidos.append((poblacion, aptitudes, k_invalido))
for pob, fits, k in casos_invalidos:
    comprobar_value_error(pob, fits, k)
print(f"Rechazo de {len(casos_invalidos)} casos invalidos: OK")

Rechazo de 20 casos invalidos: OK


## 4. Integracion real con los Roles 1 y 2

Se reutiliza `cargar_melodia_csv()` y la lectura `sf.read(..., dtype="float32")`
que emplea el Rol 2 para evaluar exactamente el WAV guardado. Su notebook
prepara ese montaje con muestras reales, las fronteras del CSV, un suavizado
de 5 ms y ganancia global. Aqui se consume el montaje existente, sin repetir
sus descargas ni escribir WAV o resultados experimentales.

El JSON debe coincidir con las notas y duraciones del CSV. No se recorta,
remuestrea ni normaliza la referencia. Si faltan archivos o dependencias,
la celda informa el problema y se detiene: no usa una referencia sustituta.
Las tres aptitudes se calculan de nuevo con `evaluar_individuo()`; no se leen
resultados guardados por el Rol 2. Ambas metricas se maximizan; nunca pasar
`error_stft_db` directamente al torneo.

In [5]:
import json
try:
    import soundfile as sf
    from src.extraccion import cargar_melodia_csv
    from src.sintesis import FRECUENCIA_MUESTREO
    from src.fitness import evaluar_individuo
except ImportError as error:
    raise ImportError("Integracion no ejecutada: faltan dependencias de requirements.txt.") from error

ruta_csv = RAIZ / "datos" / "procesados" / "melodia_01.csv"
ruta_ref = RAIZ / "datos" / "audio" / "referencias" / "flauta_referencia_01.wav"
ruta_json = ruta_ref.with_suffix(".json")
for ruta in (ruta_csv, ruta_ref, ruta_json):
    if not ruta.is_file():
        raise FileNotFoundError(f"Integracion no ejecutada; falta: {ruta.name}")

notas, duraciones = cargar_melodia_csv(ruta_csv)
metadatos = json.loads(ruta_json.read_text(encoding="utf-8"))
assert np.array_equal(notas, metadatos["notas"]), "Notas distintas del montaje"
assert np.array_equal(duraciones, metadatos["duraciones_s"]), "Duraciones distintas del montaje"
referencia, sr_ref = sf.read(ruta_ref, dtype="float32")
assert sr_ref == FRECUENCIA_MUESTREO
assert referencia.ndim == 1
assert sf.info(ruta_ref).subtype == "PCM_16"
assert referencia.size == int(round(duraciones.sum() * FRECUENCIA_MUESTREO))
assert np.all(np.isfinite(referencia)) and np.max(np.abs(referencia)) > 1e-8

rng_integracion = np.random.default_rng(2026)
poblacion_real = generar_poblacion_inicial(3, rng=rng_integracion)
aptitudes_reales = np.array([
    evaluar_individuo(gen, notas, duraciones, referencia, metrica="stft")
    for gen in poblacion_real
])
assert aptitudes_reales.shape == (3,) and np.all(np.isfinite(aptitudes_reales))
ganador_real = seleccion_torneo(poblacion_real, aptitudes_reales, k=3, rng=rng_integracion)
assert np.array_equal(ganador_real, poblacion_real[np.argmax(aptitudes_reales)])
valido, mensaje = validar_individuo(ganador_real)
assert valido, mensaje
assert ganador_real.shape == (D,) and ganador_real.dtype == np.float64
assert not np.shares_memory(ganador_real, poblacion_real)
print("Referencia:", referencia.size, "muestras a", sr_ref, "Hz")
print("Aptitudes STFT calculadas:", aptitudes_reales)
print("Indice del ganador:", int(np.argmax(aptitudes_reales)))
print("Integracion real: OK. No se escribieron audios ni resultados experimentales.")

Referencia: 59645 muestras a 22050 Hz
Aptitudes STFT calculadas: [0.67976701 0.49070326 0.70833249]
Indice del ganador: 2
Integracion real: OK. No se escribieron audios ni resultados experimentales.


## 5. Uso acordado con el Rol 4

Con la poblacion evaluada en su mismo orden, compartir el generador de la
corrida y llamar `padre = seleccion_torneo(poblacion, fitness, k=3, rng=rng)`.
La salida es solo el cromosoma, no una tupla. El ciclo, elitismo y parada
pertenecen al Rol 4. Consultar `documentacion/contratos/interfaz_operadores_rol3.md`.